# U2-03 · Del perceptrón a TensorFlow: entrenar, validar y probar

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 2, apartados 2.1 a 2.5*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). Sin GPU funciona igual; varias celdas tardan unos minutos.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges (CC BY-SA 3.0), y Fashion-MNIST, de Zalando Research (licencia MIT). Keras los descarga en la primera ejecución.

## 1. El perceptrón de Rosenblatt con MNIST

In [ ]:
import numpy as np
import keras
import matplotlib.pyplot as plt

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()

def perceptron(X, y, epocas=50):
    """Regla de Rosenblatt; y vale -1 o +1. Devuelve pesos y
    errores por época."""
    w, b, errores = np.zeros(X.shape[1]), 0.0, []
    for _ in range(epocas):
        fallos = 0
        for xi, yi in zip(X, y):
            if yi * (xi @ w + b) <= 0:      # mal clasificado
                w += yi * xi                # corrige hacia la clase
                b += yi
                fallos += 1
        errores.append(fallos)
        if fallos == 0:                     # separa todo: termina
            break
    return w, b, errores

for a, c in [(0, 1), (3, 8)]:
    sel, selp = np.isin(y_ent, [a, c]), np.isin(y_pru, [a, c])
    X = x_ent[sel].reshape(-1, 784) / 255
    y = np.where(y_ent[sel] == c, 1, -1)
    w, b, err = perceptron(X, y)
    Xp = x_pru[selp].reshape(-1, 784) / 255
    yp = np.where(y_pru[selp] == c, 1, -1)
    acierto = np.mean(np.sign(Xp @ w + b) == yp)
    print(f"{a} frente a {c}: {len(y)} imágenes, {len(err)} épocas, "
          f"fallos {err[:3]}...{err[-3:]}, prueba {acierto:.4f}")
    plt.plot(range(1, len(err) + 1), err, "o-", ms=2,
             label=f"{a} frente a {c}")
plt.yscale("symlog")
plt.xlabel("época"); plt.ylabel("errores en entrenamiento")
plt.legend(); plt.show()  # plt.savefig("perceptron.pdf")

## 2. Aproximar una función con una capa oculta (2.2)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

x = np.linspace(-3, 3, 400)[:, None].astype("float32")
f = np.sin(2 * x) + 0.3 * x ** 2              # función a imitar
fig, eje = plt.subplots(figsize=(4.4, 2.0))
eje.plot(x, f, color="0.6", lw=3, label="función")
for n in (1, 3, 10, 50):
    keras.utils.set_random_seed(0)
    red = keras.Sequential([keras.Input(shape=(1,)),
                            keras.layers.Dense(n, activation="tanh"),
                            keras.layers.Dense(1)])
    red.compile(keras.optimizers.Adam(0.01), "mse")
    red.fit(x, f, epochs=500, batch_size=64, verbose=0)
    p = red.predict(x, verbose=0)
    print(f"{n:3d} neuronas ocultas ({red.count_params():4d} "
          f"parámetros): error máximo {np.abs(p - f).max():.3f}")
    eje.plot(x, p, lw=1, label=f"{n} neuronas")
eje.legend(ncol=5, fontsize=5.5, loc="upper center")
plt.show()  # fig.savefig("aproximar.pdf")

## 3. La cinta de gradientes frente a la retropropagación a mano

Las dos primeras celdas guardan como archivos el código de los apartados 2.2 y 2.3, que la tercera importa.

In [ ]:
%%writefile l_espiral.py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier

def espiral(n=300, ruido=0.25, semilla=0):
    """Dos brazos de espiral entrelazados: clases 0 y 1."""
    rng = np.random.default_rng(semilla)
    t = np.sqrt(rng.uniform(0, 1, n)) * 3 * np.pi
    clase = rng.integers(0, 2, n)
    r = t * np.where(clase == 1, 1, -1)
    X = np.column_stack([r * np.cos(t), r * np.sin(t)]) / 3
    return X + rng.normal(0, ruido, X.shape), clase

if __name__ == "__main__":       # solo al ejecutar este archivo
    X, y = espiral()
    Xp, yp = espiral(semilla=1)            # otra muestra para probar
    xx, yy = np.meshgrid(np.linspace(-3.5, 3.5, 200),
                         np.linspace(-3.5, 3.5, 200))
    fig, ejes = plt.subplots(1, 3, figsize=(4.4, 1.6))
    for eje, n in zip(ejes, [2, 8, 64]):
        red = MLPClassifier((n,), activation="tanh", solver="lbfgs",
                            max_iter=5000,
                            random_state=0).fit(X, y)
        zona = red.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
        eje.contourf(xx, yy, zona.reshape(xx.shape),
                     levels=[0, .5, 1], colors=["#CFE0F2", "#F6DDBB"])
        eje.scatter(*X.T, c=np.where(y, "#E08A1E", "#1F5FA8"), s=2)
        eje.set_title(f"{n} neuronas", loc="center")
        eje.set_xticks([]); eje.set_yticks([])
        print(f"{n:2d} neuronas ocultas: entrenamiento "
              f"{red.score(X, y):.3f}, "
              f"prueba {red.score(Xp, yp):.3f}")
    fig.savefig("espiral.pdf")

In [ ]:
%%writefile m_retro.py
import numpy as np
from l_espiral import espiral              # datos del apartado 2.2

X, y = espiral()
y = y.reshape(-1, 1).astype(float)

def iniciar(semilla=0):
    rng = np.random.default_rng(semilla)
    return {"W1": rng.normal(0, 1, (2, 16)), "b1": np.zeros(16),
            "W2": rng.normal(0, 0.25, (16, 1)), "b2": np.zeros(1)}

def adelante(p, X):
    H = np.tanh(X @ p["W1"] + p["b1"])            # capa oculta
    P = 1 / (1 + np.exp(-(H @ p["W2"] + p["b2"])))  # salida
    return H, P

def perdida(p, X=X, y=y):
    _, P = adelante(p, X)
    return -np.mean(y * np.log(P) + (1 - y) * np.log(1 - P))

def atras(p, X=X, y=y):
    """Retropropagación: la regla de la cadena, de la salida
    hacia la entrada."""
    H, P = adelante(p, X)
    dZ2 = (P - y) / len(X)                 # error en la salida
    dH = dZ2 @ p["W2"].T                   # se reparte hacia atrás
    dZ1 = dH * (1 - H ** 2)                # derivada de tanh
    return {"W2": H.T @ dZ2, "b2": dZ2.sum(0),
            "W1": X.T @ dZ1, "b1": dZ1.sum(0)}

if __name__ == "__main__":
    p = iniciar()
    # comprobación: derivada numérica de tres pesos al azar
    g, h = atras(p), 1e-6
    for nombre, idx in [("W1", (0, 3)), ("W1", (1, 7)),
                        ("W2", (5, 0))]:
        p[nombre][idx] += h; f1 = perdida(p)
        p[nombre][idx] -= 2 * h; f2 = perdida(p)
        p[nombre][idx] += h
        print(f"{nombre}{idx}: retropropagación "
              f"{g[nombre][idx]:+.6f}, "
              f"numérica {(f1 - f2) / (2 * h):+.6f}")

    # entrenamiento: descenso de gradiente con estos gradientes
    for paso in range(1, 20_001):
        for k, v in atras(p).items():
            p[k] -= 0.5 * v
        if paso in (1, 100, 1000, 20_000):
            acierto = np.mean((adelante(p, X)[1] > 0.5) == y)
            print(f"paso {paso:5d}: pérdida {perdida(p):.4f}, "
                  f"acierto {acierto:.3f}")

In [ ]:
import numpy as np
import tensorflow as tf
from m_retro import X, y, iniciar, atras

p = iniciar()                       # los mismos pesos iniciales
v = {k: tf.Variable(a) for k, a in p.items()}
with tf.GradientTape() as cinta:    # TensorFlow anota cada paso
    H = tf.tanh(X @ v["W1"] + v["b1"])
    logit = H @ v["W2"] + v["b2"]
    perdida = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(
        labels=y, logits=logit))
g_tf = cinta.gradient(perdida, v)           # y deriva hacia atrás

g_np = atras(p)
for k in p:
    dif = np.max(np.abs(g_tf[k].numpy() - g_np[k]))
    print(f"{k}: diferencia máxima {dif:.1e}")

## 4. El gradiente que se desvanece, medido (2.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

(x, y), _ = keras.datasets.fashion_mnist.load_data()
xb = tf.constant(x[:256].reshape(256, -1) / 255, tf.float32)
yb = tf.constant(y[:256])
perdida = keras.losses.SparseCategoricalCrossentropy()
plt.figure(figsize=(4.4, 2.0))
for act in ("sigmoid", "relu"):
    keras.utils.set_random_seed(0)
    capas = [keras.layers.Dense(64, activation=act)
             for _ in range(10)]
    red = keras.Sequential([keras.Input(shape=(784,)), *capas,
                            keras.layers.Dense(10, "softmax")])
    with tf.GradientTape() as cinta:
        p = perdida(yb, red(xb))
    g = cinta.gradient(p, [c.kernel for c in capas])
    normas = [float(tf.norm(gi)) for gi in g]
    print(f"{act:<7} norma del gradiente, capa 1: {normas[0]:.1e},"
          f" mín. {min(normas):.1e}, máx. {max(normas):.1e},"
          f" capa 10: {normas[-1]:.1e}")
    plt.semilogy(range(1, 11), normas, "o-", label=act)
plt.xlabel("capa oculta"); plt.ylabel("norma del gradiente")
plt.legend(); plt.show()  # plt.savefig("desvanecer.pdf")

## 5. Por qué no se derivan los pesos uno a uno (2.3)

In [ ]:
import time
import numpy as np
import tensorflow as tf

rng = np.random.default_rng(0)
X = tf.constant(rng.normal(size=(64, 100)), tf.float32)
for n in (100, 400, 1600):
    W = tf.Variable(rng.normal(0, 0.1, (100, n)), dtype=tf.float32)
    V = tf.constant(rng.normal(0, 0.1, (n, 1)), tf.float32)
    f = lambda: tf.reduce_mean(tf.tanh(X @ W) @ V)
    for _ in range(2):                  # se mide la segunda vez
        t = time.perf_counter()
        with tf.GradientTape() as cinta:
            valor = f()
        g = cinta.gradient(valor, W)
        t_auto = time.perf_counter() - t
    t = time.perf_counter(); h = 1e-3          # diferencias finitas
    for k in range(20):                         # solo 20 pesos
        i, j = divmod(k, n)
        W[i, j].assign(W[i, j] + h); arriba = f()
        W[i, j].assign(W[i, j] - h)
        numerica = (arriba - valor) / h
    t_20 = time.perf_counter() - t
    print(f"{100 * n:7d} pesos: retropropagación {t_auto * 1e3:4.1f}"
          f" ms; numérico estimado {t_20 / 20 * 100 * n:5.1f} s")
    print(f"   un peso: {g[0, 19].numpy():+.5f} (cinta) "
          f"{numerica:+.5f} (numérico)")

## 6. Fashion-MNIST y cinco decisiones de entrenamiento

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x, y), _ = keras.datasets.fashion_mnist.load_data()
clases = ["camiseta", "pantalón", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botín"]
fig, ejes = plt.subplots(1, 10, figsize=(4.4, 0.75))
for c, eje in enumerate(ejes):
    eje.imshow(x[np.flatnonzero(y == c)[0]])
    eje.set_title(clases[c], fontsize=5, loc="center",
                  fontweight="normal")
    eje.axis("off")
plt.show()  # fig.savefig("fashion.pdf")
print(x.shape, np.bincount(y))

In [ ]:
import time
import numpy as np
import keras

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x / 255.0, x_pru / 255.0
x_ent, y_ent = x[:50_000], y[:50_000]       # entrenamiento
x_val, y_val = x[50_000:], y[50_000:]       # validación

def entrenar(optimizador, epocas=10, lote=32, inicial="he_normal"):
    keras.utils.set_random_seed(0)
    red = keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu",
                           kernel_initializer=inicial),
        keras.layers.Dense(128, activation="relu",
                           kernel_initializer=inicial),
        keras.layers.Dense(10, activation="softmax")])
    red.compile(optimizador, "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    t0 = time.perf_counter()
    h = red.fit(x_ent, y_ent, epochs=epocas, batch_size=lote,
                validation_data=(x_val, y_val), verbose=0)
    return red, h.history, time.perf_counter() - t0

opt = keras.optimizers
print("1) tasa de aprendizaje (SGD, 10 épocas)")
for eta in [0.001, 0.01, 0.1, 1.0]:
    _, h, _ = entrenar(opt.SGD(eta))
    print(f"   eta = {eta:<5}: "
          f"validación {h['val_accuracy'][-1]:.4f}")

print("2) optimizador (10 épocas)")
curvas = {}
for nombre, o in [("SGD", opt.SGD(0.01)),
                  ("SGD + momento", opt.SGD(0.01, momentum=0.9)),
                  ("Adam", opt.Adam(0.001))]:
    _, h, s = entrenar(o)
    curvas[nombre] = h["val_accuracy"]
    print(f"   {nombre:<14}: validación {h['val_accuracy'][-1]:.4f}"
          f" ({s:.0f} s)")

print("3) tamaño de lote (Adam, 10 épocas)")
for lote in [16, 128, 1024]:
    _, h, s = entrenar(opt.Adam(0.001), lote=lote)
    print(f"   lote {lote:4d}: validación "
          f"{h['val_accuracy'][-1]:.4f}, {s:.0f} s")

print("4) pesos iniciales (Adam, 5 épocas)")
for inicial in ["zeros", "he_normal"]:
    _, h, _ = entrenar(opt.Adam(0.001), epocas=5, inicial=inicial)
    print(f"   {inicial:<9}: validación {h['val_accuracy'][-1]:.4f}")
np.savez("curvas_opt.npz", **curvas)

**Prueba.** Repite la comparación de optimizadores con 20 épocas. ¿Sigue sin ayudar el momento?

## 7. Parada temprana y evaluación final

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x / 255.0, x_pru / 255.0
clases = ["camiseta", "pantalón", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botín"]

keras.utils.set_random_seed(0)
red = keras.Sequential([
    keras.Input(shape=(28, 28)), keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax")])
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
parada = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)
h = red.fit(x[:50_000], y[:50_000], epochs=60,
            validation_data=(x[50_000:], y[50_000:]),
            callbacks=[parada], verbose=0)
mejor = int(np.argmin(h.history["val_loss"]))
print(f"se detiene en la época {len(h.history['loss'])}; "
      f"la mejor fue la {mejor + 1}")
print(f"pérdida en la mejor época: entrenamiento "
      f"{h.history['loss'][mejor]:.3f}, "
      f"validación {h.history['val_loss'][mejor]:.3f}")

# una sola evaluación final, en prueba
perdida, acierto = red.evaluate(x_pru, y_pru, verbose=0)
print(f"prueba: acierto {acierto:.4f}")
pred = red.predict(x_pru, verbose=0).argmax(1)
for c in range(10):
    print(f"  {clases[c]:<10} {np.mean(pred[y_pru == c] == c):.3f}")

epocas = np.arange(1, len(h.history["loss"]) + 1)
plt.plot(epocas, h.history["loss"], label="entrenamiento")
plt.plot(epocas, h.history["val_loss"], label="validación")
plt.axvline(mejor + 1, color="#9AA5B1", ls="--")
plt.xlabel("época"); plt.ylabel("pérdida (entropía cruzada)")
plt.legend(); plt.show()  # plt.savefig("parada.pdf")

## 8. Programar la tasa de aprendizaje

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
pasos = 15 * (50_000 // 128 + 1)                 # 15 épocas
programas = {
    "fija 0,001": 0.001,
    "fija 0,01": 0.01,
    "coseno desde 0,01": keras.optimizers.schedules.CosineDecay(
        0.01, pasos, alpha=0.01),
    "calentamiento + coseno": keras.optimizers.schedules.CosineDecay(
        0.0, pasos - 1000, alpha=0.01, warmup_target=0.01,
        warmup_steps=1000)}
plt.figure(figsize=(4.4, 2.0))
for nombre, tasa in programas.items():
    keras.utils.set_random_seed(0)
    red = keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])
    red.compile(keras.optimizers.Adam(tasa),
                "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    h = red.fit(x[:50_000], y[:50_000], epochs=15, batch_size=128,
                validation_data=(x[50_000:], y[50_000:]), verbose=0)
    v = h.history["val_accuracy"]
    print(f"{nombre:<23} validación final {v[-1]:.4f}, mejor "
          f"{max(v):.4f}")
    if callable(tasa):
        plt.plot([float(tasa(i)) for i in range(0, pasos, 50)],
                 label=nombre)
plt.xlabel("paso (×50)"); plt.ylabel("tasa"); plt.legend()
plt.show()  # plt.savefig("programas.pdf")

## 9. ¿Más datos o más modelo?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x, y), _ = keras.datasets.fashion_mnist.load_data()
x = x / 255.0
xv, yv = x[50_000:], y[50_000:]
tamanos = [1000, 3000, 10_000, 30_000, 50_000]
ent, val = [], []
for n in tamanos:
    keras.utils.set_random_seed(0)
    red = keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])
    red.compile("adam", "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    red.fit(x[:n], y[:n], epochs=max(5, 150_000 // n),
            batch_size=128, verbose=0)   # al menos 150 000 ejemplos
    ent.append(red.evaluate(x[:n], y[:n], verbose=0)[1])
    val.append(red.evaluate(xv, yv, verbose=0)[1])
    print(f"{n:6d} prendas: entrenamiento {ent[-1]:.3f}, "
          f"validación {val[-1]:.3f}, "
          f"diferencia {ent[-1] - val[-1]:.3f}")
plt.figure(figsize=(4.4, 2.0))
plt.semilogx(tamanos, ent, "o-", label="entrenamiento")
plt.semilogx(tamanos, val, "o-", label="validación")
plt.xlabel("prendas de entrenamiento"); plt.ylabel("acierto")
plt.legend(); plt.show()  # plt.savefig("curvas_aprendizaje.pdf")

## 10. Validación cruzada

In [ ]:
import numpy as np
import keras
from sklearn.model_selection import StratifiedKFold

(x, y), _ = keras.datasets.fashion_mnist.load_data()
x, y = x[:10_000] / 255.0, y[:10_000]       # un problema pequeño

def red():
    return keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])

aciertos = []
pliegues = StratifiedKFold(5, shuffle=True, random_state=0)
for k, (e, v) in enumerate(pliegues.split(x, y), start=1):
    keras.utils.set_random_seed(0)
    m = red()
    m.compile("adam", "sparse_categorical_crossentropy",
              metrics=["accuracy"])
    m.fit(x[e], y[e], epochs=10, batch_size=128, verbose=0)
    aciertos.append(m.evaluate(x[v], y[v], verbose=0)[1])
    print(f"pliegue {k}: {aciertos[-1]:.4f}")
print(f"media {np.mean(aciertos):.4f} ± "
      f"{np.std(aciertos, ddof=1):.4f}")

## 11. Normalizar dentro de la red

In [ ]:
import time
import keras
from keras import layers

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0

def profunda(normalizar, act):
    keras.utils.set_random_seed(0)
    capas = [keras.Input(shape=(28, 28)), layers.Flatten()]
    for _ in range(10):                      # diez capas ocultas
        capas.append(layers.Dense(64))
        if normalizar:
            capas.append(layers.BatchNormalization())
        capas.append(layers.Activation(act))
    red = keras.Sequential(capas + [layers.Dense(10, "softmax")])
    red.compile("adam", "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    t = time.perf_counter()
    h = red.fit(x[:50_000], y[:50_000], epochs=5, batch_size=128,
                validation_data=(x[50_000:], y[50_000:]), verbose=0)
    return h.history["val_accuracy"], time.perf_counter() - t

for act in ("sigmoid", "relu"):
    for norm in (False, True):
        v, t = profunda(norm, act)
        print(f"{act:<7} normalización {norm!s:<5}: validación por "
              f"época {[round(a, 3) for a in v]} ({t:.0f} s)")

## 12. ¿Cuánto cambia un resultado por azar?

In [ ]:
import numpy as np
import keras

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
val = []
for semilla in range(5):
    keras.utils.set_random_seed(semilla)
    red = keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10, activation="softmax")])
    red.compile("adam", "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    h = red.fit(x[:50_000], y[:50_000], epochs=5, batch_size=128,
                validation_data=(x[50_000:], y[50_000:]), verbose=0)
    val.append(h.history["val_accuracy"][-1])
    print(f"semilla {semilla}: validación {val[-1]:.4f}")
print(f"media {np.mean(val):.4f}, desviación "
      f"{np.std(val, ddof=1):.4f}, rango {max(val) - min(val):.4f}")

## 13. Búsqueda aleatoria de hiperparámetros

In [ ]:
import numpy as np
import keras

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
xe, ye, xv, yv = x[:50_000], y[:50_000], x[50_000:], y[50_000:]

def prueba(tasa, neuronas, dropout, semilla=0):
    keras.utils.set_random_seed(semilla)
    red = keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(neuronas, activation="relu"),
        keras.layers.Dropout(dropout),
        keras.layers.Dense(10, activation="softmax")])
    red.compile(keras.optimizers.Adam(tasa),
                "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    red.fit(xe, ye, epochs=3, batch_size=128, verbose=0)
    return red, red.evaluate(xv, yv, verbose=0)[1]

rng = np.random.default_rng(1)                # búsqueda aleatoria
resultados = []
for i in range(8):
    conf = (float(10 ** rng.uniform(-4, -1.5)),     # tasa
            int(rng.choice([32, 64, 128, 256, 512])),
            float(rng.choice([0.0, 0.2, 0.4])))
    red, acierto = prueba(*conf)
    resultados.append((acierto, conf, red))
    print(f"tasa {conf[0]:.5f}, {conf[1]:3d} neuronas, dropout "
          f"{conf[2]}: validación {acierto:.4f}")
acierto, conf, red = max(resultados, key=lambda r: r[0])
print(f"elegida: tasa {conf[0]:.5f}, {conf[1]} neuronas, dropout "
      f"{conf[2]} | prueba {red.evaluate(xp, yp, verbose=0)[1]:.4f}")

## 14. Tres maneras de escribir la red y guardarla

In [ ]:
import numpy as np
import tensorflow as tf
import keras

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x.astype("float32") / 255, x_pru.astype("float32") / 255

# 1) tensores y variables
t = tf.constant([[1.0, 2.0], [3.0, 4.0]])
w = tf.Variable(tf.ones((2, 1)))
print("tensor", t.shape, t.dtype,
      "| t @ w =", (t @ w).numpy().ravel())

# 2) la misma red, de tres formas
def secuencial():
    return keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10)])

def funcional():
    entrada = keras.Input(shape=(28, 28))
    h = keras.layers.Flatten()(entrada)
    h = keras.layers.Dense(128, activation="relu")(h)
    return keras.Model(entrada, keras.layers.Dense(10)(h))

class Subclase(keras.Model):
    def __init__(self):
        super().__init__()
        self.aplanar = keras.layers.Flatten()
        self.oculta = keras.layers.Dense(128, activation="relu")
        self.salida = keras.layers.Dense(10)
    def call(self, entrada):
        return self.salida(self.oculta(self.aplanar(entrada)))

perdida = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
for nombre, crear in [("Sequential", secuencial),
                      ("funcional", funcional),
                      ("subclase", Subclase)]:
    keras.utils.set_random_seed(0)
    red = crear()
    red.compile("adam", perdida, metrics=["accuracy"])
    red.fit(x, y, epochs=3, verbose=0)
    print(f"{nombre:<10}: {red.count_params():,} parámetros, prueba "
          f"{red.evaluate(x_pru, y_pru, verbose=0)[1]:.4f}")

# 3) el mismo entrenamiento con un bucle propio
keras.utils.set_random_seed(0)
red, optim = secuencial(), keras.optimizers.Adam()
datos = tf.data.Dataset.from_tensor_slices((x, y)).shuffle(
    60_000, seed=0).batch(32)
for epoca in range(3):
    for xb, yb in datos:
        with tf.GradientTape() as cinta:
            p = perdida(yb, red(xb, training=True))
        grads = cinta.gradient(p, red.trainable_variables)
        optim.apply_gradients(zip(grads, red.trainable_variables))
acierto = np.mean(red.predict(x_pru, verbose=0).argmax(1) == y_pru)
print(f"bucle propio: prueba {acierto:.4f}")

# 4) guardar, cargar y comprobar
red.save("ropa.keras")
cargada = keras.models.load_model("ropa.keras")
iguales = np.allclose(red.predict(x_pru[:100], verbose=0),
                      cargada.predict(x_pru[:100], verbose=0))
print("el modelo cargado predice lo mismo:", iguales)

## 15. Una red con dos salidas

In [ ]:
import numpy as np
import keras
from keras import layers

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
grupo = np.array([0, 1, 0, 0, 0, 2, 0, 2, 3, 2])   # ropa de arriba,
g, gp = grupo[y], grupo[yp]       # pantalón, calzado, bolso

keras.utils.set_random_seed(0)
e = keras.Input(shape=(28, 28), name="imagen")
h = layers.Dense(256, activation="relu")(layers.Flatten()(e))
prenda = layers.Dense(10, activation="softmax", name="prenda")(h)
tipo = layers.Dense(4, activation="softmax", name="grupo")(h)
red = keras.Model(e, [prenda, tipo])             # dos salidas
red.compile("adam",
            loss={"prenda": "sparse_categorical_crossentropy",
                  "grupo": "sparse_categorical_crossentropy"},
            loss_weights={"prenda": 1.0, "grupo": 0.5},
            metrics={"prenda": ["accuracy"], "grupo": ["accuracy"]})
red.fit(x, {"prenda": y, "grupo": g}, epochs=10, batch_size=128,
        verbose=0)
pp, pg = red.predict(xp, verbose=0)
print(f"acierto prenda {np.mean(pp.argmax(1) == yp):.4f}, "
      f"grupo {np.mean(pg.argmax(1) == gp):.4f}")
coherente = grupo[pp.argmax(1)] == pg.argmax(1)
print(f"respuestas coherentes entre las dos salidas: "
      f"{coherente.mean():.1%}")

## 16. Capas y pérdidas propias

In [ ]:
import numpy as np
import keras
from keras import layers, ops

class Escala(layers.Layer):          # una capa propia con pesos
    def build(self, forma):
        self.a = self.add_weight(shape=(forma[-1],),
                                 initializer="ones", name="a")
        self.b = self.add_weight(shape=(forma[-1],),
                                 initializer="zeros", name="b")
    def call(self, x):
        return x * self.a + self.b   # a y b se aprenden

def perdida_suave(y, p):             # una pérdida propia
    y1 = ops.one_hot(ops.cast(y, "int32"), 10)
    y1 = y1 * 0.9 + 0.01             # etiquetas suavizadas
    return keras.losses.categorical_crossentropy(y1, p)

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0
keras.utils.set_random_seed(0)
red = keras.Sequential([keras.Input(shape=(28, 28)),
                        layers.Flatten(), Escala(),
                        layers.Dense(128, activation="relu"),
                        layers.Dense(10, activation="softmax")])
red.compile("adam", perdida_suave, metrics=["accuracy"])
red.fit(x, y, epochs=5, batch_size=128, verbose=0)
print("prueba:", round(red.evaluate(xp, yp, verbose=0)[1], 4))
a = red.layers[1].get_weights()[0]
print("pesos a aprendidos: mín.", round(a.min(), 3), "máx.",
      round(a.max(), 3), "| parámetros de Escala:", 2 * a.size)
print("confianza media:",
      round(red.predict(xp, verbose=0).max(1).mean(), 3))

## 17. Callbacks en acción

In [ ]:
import numpy as np
import keras

(x, y), (xp, yp) = keras.datasets.fashion_mnist.load_data()
x, xp = x / 255.0, xp / 255.0

class Informe(keras.callbacks.Callback):      # un callback propio
    tasa = None
    def on_epoch_end(self, epoca, logs=None):
        tasa = float(self.model.optimizer.learning_rate)
        if tasa != self.tasa:                  # solo si cambia
            print(f"época {epoca + 1:2d}: validación "
                  f"{logs['val_accuracy']:.4f}, tasa {tasa:.5f}")
            self.tasa = tasa
    def on_train_end(self, logs=None):
        h = self.model.history.history["val_accuracy"]
        print(f"parada en la época {len(h)}; mejor validación "
              f"{max(h):.4f} (época {h.index(max(h)) + 1})")

keras.utils.set_random_seed(0)
red = keras.Sequential([
    keras.Input(shape=(28, 28)), keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(10, activation="softmax")])
red.compile(keras.optimizers.Adam(0.003),
            "sparse_categorical_crossentropy", metrics=["accuracy"])
avisos = [
    keras.callbacks.ModelCheckpoint("mejor.keras",
                                    monitor="val_accuracy",
                                    save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss",
                                      factor=0.3, patience=2,
                                      min_lr=1e-4),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4),
    keras.callbacks.CSVLogger("registro.csv"), Informe()]
red.fit(x[:50_000], y[:50_000], epochs=40, batch_size=128,
        validation_data=(x[50_000:], y[50_000:]),
        callbacks=avisos, verbose=0)
mejor = keras.models.load_model("mejor.keras")
print("mejor modelo guardado, en prueba:",
      round(mejor.evaluate(xp, yp, verbose=0)[1], 4))

## 18. Datos en flujo con tf.data

In [ ]:
import time
import tensorflow as tf
import keras

(x, y), _ = keras.datasets.fashion_mnist.load_data()

def preparar(img, etiqueta):        # se aplica a cada ejemplo
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.random_flip_left_right(img[..., None])
    return img, etiqueta

datos = (tf.data.Dataset.from_tensor_slices((x, y))
         .shuffle(10_000, seed=0)
         .map(preparar, num_parallel_calls=tf.data.AUTOTUNE)
         .batch(128)
         .prefetch(tf.data.AUTOTUNE))   # prepara el lote siguiente
for img, et in datos.take(1):
    print("un lote:", img.shape, img.dtype, et.shape)
t = time.perf_counter()
n = sum(int(et.shape[0]) for _, et in datos)
print(f"una pasada completa: {n} imágenes en "
      f"{time.perf_counter() - t:.1f} s")

**Ejercicio.** Añade una tercera capa oculta de 64 neuronas a la versión funcional y entrénala con parada temprana. ¿Mejora la camisa?

> Xiao, H., Rasul, K. y Vollgraf, R. (2017). Fashion-MNIST: a novel image dataset for benchmarking machine learning algorithms. arXiv:1708.07747. https://doi.org/10.48550/arXiv.1708.07747
>
> Chollet, F. (2021). *Deep Learning with Python* (2.ª ed.). Manning. ISBN 978-1-61729-686-4